# Kantorovich--Rubinstein potential for $\Wass_1$

This notebook generates `fig:w1-kantorovich-rubinstein-potential`.  For the signed measure $\alpha-\beta$ on the line, the Kantorovich--Rubinstein formula reads
$$
\Wass_1(\alpha,\beta)=\max_{\operatorname{Lip}(f)\leq 1}\int f\,d(\alpha-\beta).
$$
We place the red atoms of $\alpha$ at the two extremes and the blue atoms of $\beta$ near the center.  The V-shaped function $f(x)=|x|$ is 1-Lipschitz and is optimal: along every transported pair $(x_i,y_j)$ carrying mass, $f(x_i)-f(y_j)=|x_i-y_j|$.

In [ ]:
from pathlib import Path
import sys
import contextlib
import io

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.patches import FancyArrowPatch

from figure_style import (
    AXIS_LINE_WIDTH,
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    MASS_MARKER_MAX_FACTOR,
    MASS_MARKER_MIN_FACTOR,
    POINT_EDGE_WIDTH,
    RED,
    TRANSPORT_LINE_MAX_WIDTH,
    TRANSPORT_LINE_MIN_WIDTH,
    VIOLET,
    box_axes,
    figure_dir,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()


def quiet_save_pdf(fig, path, *, pad_inches=0.035):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        save_pdf(fig, path, pad_inches=pad_inches)

## A four-atom one-dimensional transport problem

The source atoms are farther from the origin than the target atoms.  In one dimension the optimal plan for the cost $|x-y|$ is monotone, and here POT's linear program recovers the sorted inward transport.  The dual equality check below verifies that the displayed witness reaches the primal cost.

In [ ]:
fig_name = "w1-kantorovich-rubinstein-potential"
out = figure_dir(fig_name)

alpha_x = np.array([-1.85, -1.28, 1.28, 1.85])
beta_x = np.array([-0.72, -0.25, 0.25, 0.72])
a = np.ones(len(alpha_x)) / len(alpha_x)
b = np.ones(len(beta_x)) / len(beta_x)

C = np.abs(alpha_x[:, None] - beta_x[None, :])
plan = ot.emd(a, b, C)

potential = np.abs
dual_value = float(np.dot(a, potential(alpha_x)) - np.dot(b, potential(beta_x)))
primal_value = float(np.sum(plan * C))
assert np.allclose(primal_value, dual_value, atol=1e-12)

pairs = []
for i, j in zip(*np.nonzero(plan > 1e-12)):
    pairs.append((int(i), int(j), float(plan[i, j])))

primal_value, dual_value, pairs

## Exported panels

The first panel shows the 1-Lipschitz witness.  Violet portions of the graph are exactly the intervals used by the optimal transport plan.  The second panel isolates the saturated transport directions; arrows point from red source atoms to blue target atoms, with widths proportional to transported mass.

In [ ]:
def width_from_masses(masses, max_width=TRANSPORT_LINE_MAX_WIDTH, min_width=TRANSPORT_LINE_MIN_WIDTH):
    masses = np.asarray(masses, dtype=float)
    masses = masses / max(float(masses.max()), 1e-15)
    return min_width + (max_width - min_width) * np.sqrt(masses)

# Potential witness.
grid = np.linspace(-2.08, 2.08, 600)
fig, ax = plt.subplots(figsize=(3.45, 2.05))
ax.plot(grid, potential(grid), color=GRAY, lw=1.15, zorder=1)

for i, j, mass in pairs:
    lo, hi = sorted([alpha_x[i], beta_x[j]])
    local_grid = np.linspace(lo, hi, 160)
    ax.plot(local_grid, potential(local_grid), color=VIOLET, lw=2.05, alpha=0.58, solid_capstyle="round", zorder=2)

for x0 in alpha_x:
    ax.plot([x0, x0], [0, potential(x0)], color=RED, lw=0.55, alpha=0.18, zorder=0)
for y0 in beta_x:
    ax.plot([y0, y0], [0, potential(y0)], color=BLUE, lw=0.55, alpha=0.18, zorder=0)

ax.scatter(alpha_x, potential(alpha_x), s=DIRAC_MARKER_SIZE, color=RED, edgecolor="none", linewidth=POINT_EDGE_WIDTH, zorder=4)
ax.scatter(beta_x, potential(beta_x), s=DIRAC_MARKER_SIZE, color=BLUE, edgecolor="none", linewidth=POINT_EDGE_WIDTH, zorder=4)
ax.set_xlim(-2.05, 2.05)
ax.set_ylim(-0.08, 2.05)
ax.set_xlabel(r"$x$", labelpad=1)
ax.set_ylabel(r"$f(x)$", labelpad=2)
ax.set_xticks([-2, -1, 0, 1, 2])
ax.set_yticks([0, 1, 2])
box_axes(ax)
quiet_save_pdf(fig, out / "potential.pdf", pad_inches=0.045)
plt.close(fig)

# Saturated transport directions.
fig, ax = plt.subplots(figsize=(3.45, 1.05))
ax.axhline(0.0, color=LIGHT_GRAY, lw=0.85, zorder=0)
widths = width_from_masses([mass for _, _, mass in pairs], max_width=1.45, min_width=0.55)
for k, (i, j, mass) in enumerate(pairs):
    x0, x1 = alpha_x[i], beta_x[j]
    rad = 0.18 + 0.055 * (k % 2)
    arrow = FancyArrowPatch(
        (x0, 0.03),
        (x1, 0.03),
        arrowstyle="-|>",
        mutation_scale=7.8,
        connectionstyle=f"arc3,rad={rad}",
        linewidth=widths[k],
        color=VIOLET,
        alpha=0.72,
        shrinkA=5.5,
        shrinkB=5.5,
        zorder=2,
    )
    ax.add_patch(arrow)

ax.scatter(alpha_x, np.zeros_like(alpha_x), s=DIRAC_MARKER_SIZE, color=RED, edgecolor="none", linewidth=POINT_EDGE_WIDTH, zorder=4)
ax.scatter(beta_x, np.zeros_like(beta_x), s=DIRAC_MARKER_SIZE, color=BLUE, edgecolor="none", linewidth=POINT_EDGE_WIDTH, zorder=4)
ax.set_xlim(-2.05, 2.05)
ax.set_ylim(-0.18, 0.58)
remove_axes(ax)
quiet_save_pdf(fig, out / "transport.pdf", pad_inches=0.055)
plt.close(fig)